# Lab 4 - Baskteball Database

## Importing Libraries

In [1]:
import psycopg2  #import of the psycopg2 python library
import pandas as pd #import of the pandas python library
import pandas.io.sql as psql

##No transaction is started when commands are executed and no commit() or rollback() is required. 
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

## Connecting to the Default DB

In [2]:
try:
    # Connect to the postgreSQL server with username, and password credentials
    con = psycopg2.connect(user = "postgres",
                                  password = "postgres",
                                  host = "postgres",
                                  port = "5432")
    
    con.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT);
    print("Connected Successfully to PostgreSQL server!!")
    
    # Obtain a DB Cursor to perform database operations
    cursor = con.cursor();
except (Exception, psycopg2.Error) as error :
     print ("Error while connecting to PostgreSQL", error)


Connected Successfully to PostgreSQL server!!


## Create the database that we will use during this practice

In [35]:
#DB_name variable    

# Create DB statement
sqlCreateDatabase = "CREATE DATABASE basketball;"

try:
    # Execute a SQL command: this creates a new DB
    cursor.execute(sqlCreateDatabase);
    print("Database basketball Created Successfully!")
except (Exception, psycopg2.Error) as error :
    print("Error While Creating the DB: ",error)
    
finally:
    # Close communication with the database
    cursor.close() #to close the cusrsor
    con.close() #to close the connection/ we will open a new connection to the created DB

Error While Creating the DB:  current transaction is aborted, commands ignored until end of transaction block



### We can verify if the database exist on pgAdmin, exposed on the same code-space on port 8080.


### Connect the the created tabase locally, so we can interact with the database

In [110]:
# get a new connection but this time point to the created "tartupurchases" DB.
con = psycopg2.connect(user = "postgres",
                       password = "postgres",
                       host = "postgres",
                       port = "5432",
                       database = "basketball")

try:
    # Obtain a new DB Cursor (to "tartupurchases" DB )
    cursor = con.cursor();
    print("connected again to the server and cusor now on basketball DB !!")
except (Exception, psycopg2.Error) as error:
    print("Error in Connection",error)

connected again to the server and cusor now on basketball DB !!


<connection object at 0x77f84f175e40; dsn: 'user=postgres password=xxx dbname=basketball host=postgres port=5432', closed: 0>

# Database Creation

Our initiation script is stored in a file called 'basketball_practice_data.sql'

How do I load it from python? that is easy, we just open the file as a stream.

![](./basketball_db_schema.png)

In [15]:
mydb = open("basketball_practice_data.sql", "r").read() # the .sql file is in the same path of this notebook

In [10]:
print(mydb)


-- Basketball Practice Dataset for Postgres
-- Schema + INSERTs
-- Run as a single script.

-- ============================
-- DDL (idempotent, safe re-run)
-- ============================

CREATE TABLE IF NOT EXISTS team (
    team_id      SERIAL PRIMARY KEY,
    name         TEXT NOT NULL UNIQUE,
    city         TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS coach (
    coach_id     SERIAL PRIMARY KEY,
    full_name    TEXT NOT NULL,
    role         TEXT NOT NULL DEFAULT 'Head Coach',
    team_id      INTEGER REFERENCES team(team_id) ON DELETE SET NULL
);

CREATE TABLE IF NOT EXISTS player (
    player_id    SERIAL PRIMARY KEY,
    first_name   TEXT NOT NULL,
    last_name    TEXT NOT NULL,
    email        TEXT UNIQUE,
    gender       CHAR(1) CHECK (gender IN ('M','F')),
    age          INTEGER CHECK (age > 0),
    position     TEXT CHECK (position IN ('Guard','Forward','Center','Coach Assistant')),
    avg_points   NUMERIC(5,2) DEFAULT 0,
    city         TEXT,
    state        

In [11]:
try:

    #Execute this command (SQL Query)
    cursor.execute(mydb) #passing the file content to the execute
    
    # Make the changes to the database persistent
    con.commit()
    print("Tables from input file were created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    # if it exits with an exception the transaction is rolled back.
    con.rollback()
    print("Error While Creating the DB: ",error)

Tables from input file were created successfully in PostgreSQL 


We can check on pgAdmin that everything is fine, or...

In [25]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""SELECT table_name 
                  FROM information_schema.tables 
                  WHERE table_schema = 'public'  
               """)

for table in cursor.fetchall():
    print(table)

('team',)
('coach',)
('player',)
('match',)
('match_player',)


In [27]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""SELECT * 
                  FROM player 
                  LIMIT 10;
               """)

for table in cursor.fetchall():
    print(table)

(1, 'Luca', 'Moretti', 'luca.moretti@nba.com', 'M', 21, 'Guard', Decimal('14.20'), 'Lyon', 'ARA', 1)
(2, 'Noah', 'Martin', 'noah.martin@gmail.com', 'M', 24, 'Forward', Decimal('11.30'), 'Lyon', 'ARA', 1)
(3, 'Ethan', 'Durand', 'ethan.durand@yahoo.com', 'M', 22, 'Guard', Decimal('9.70'), 'Villeurbanne', 'ARA', 1)
(4, 'Adam', 'Nguyen', 'adam.nguyen@outlook.com', 'M', 20, 'Center', Decimal('6.50'), 'Lyon', 'ARA', 1)
(5, 'Sofia', 'Bernard', 'sofia.bernard@gmail.com', 'F', 23, 'Forward', Decimal('12.90'), 'Lyon', 'ARA', 1)
(6, 'Amine', 'Khelifi', 'amine.khelifi@nba.com', 'M', 25, 'Guard', Decimal('8.40'), 'Bron', 'ARA', 1)
(7, 'Julie', 'Roux', 'julie.roux@gmail.com', 'F', 22, 'Coach Assistant', Decimal('0.00'), 'Lyon', 'ARA', 1)
(8, 'Milan', 'Zoric', 'milan.zoric@nba.com', 'M', 19, 'Guard', Decimal('7.20'), 'Lyon', 'ARA', 1)
(9, 'Yanis', 'Haddad', 'yanis.haddad@nba.com', 'M', 26, 'Forward', Decimal('13.10'), 'Marseille', 'PACA', 2)
(10, 'Mehdi', 'Talbi', 'mehdi.talbi@gmail.com', 'M', 24, 'G

## EXERCISE 1

### 1:  Create a view named team_summary allowing the sports director to verify team identities (team name, city, coach). Check view content and drop it.

In [38]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""CREATE OR REPLACE VIEW team_summary AS
                  SELECT t.name, t.city, c.full_name as coach
                  FROM coach  c JOIN team t ON t.team_id = c.team_id
                  LIMIT 10;
               """)



In [39]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""SELECT table_name 
                  FROM information_schema.views 
                  WHERE table_schema = 'public'  
               """)

for table in cursor.fetchall():
    print(table)

('team_summary',)


In [40]:
cursor.execute("""SELECT * 
                  FROM team_summary  
               """)

for table in cursor.fetchall():
    print(table)

('Lyon Eagles', 'Lyon', 'Jean Dupont')
('Marseille Sharks', 'Marseille', 'Amina Bensaid')
('Paris Titans', 'Paris', 'Thomas Leroy')
('Shelbyville Bears', 'Shelbyville', 'Clara Moreau')
('Greenville Wolves', 'Greenville', 'Marco Rossi')


## 2. Create high_scorers view on the Player table grouping the players  whose average points per game are higher than the overall average and whose position corresponds to 'Guard'. Check your results.


In [55]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""CREATE OR REPLACE VIEW high_scorers AS
                  SELECT p.player_id, p.first_name, p.last_name, AVG(s.points) as avg_points
                  FROM player p JOIN match_player s ON p.player_id = s.player_id
                  WHERE p.position = 'Guard' 
                  GROUP BY p.player_id, p.first_name, p.last_name
                  HAVING AVG(s.points) > (SELECT AVG(s2.points) FROM match_player s2 )
               """)


In [56]:
cursor.execute("""SELECT * 
                  FROM high_scorers  
               """)

for table in cursor.fetchall():
    print(table)

(1, 'Luca', 'Moretti', Decimal('13.6666666666666667'))
(3, 'Ethan', 'Durand', Decimal('13.6666666666666667'))
(6, 'Amine', 'Khelifi', Decimal('13.6666666666666667'))
(8, 'Milan', 'Zoric', Decimal('13.6666666666666667'))
(17, 'Pierre', 'Leclerc', Decimal('14.5000000000000000'))
(21, 'Elise', 'Laurent', Decimal('14.5000000000000000'))
(22, 'Ibrahim', 'Sow', Decimal('14.5000000000000000'))


### -- 3. Create young_players view on the Player table grouping players whose age is less than 23.  The insertion or update on this view must always respect this constraint.

In [71]:
cursor.execute("""CREATE OR REPLACE VIEW young_players AS
                  SELECT p.player_id, p.first_name, p.last_name, p.age
                  FROM player p 
                  WHERE p.age < 23
                  WITH CHECK OPTION
                  
               """)

In [62]:
cursor.execute("""SELECT * 
                  FROM young_players  
               """)

for table in cursor.fetchall():
    print(table)

(1, 'Luca', 'Moretti', 21)
(3, 'Ethan', 'Durand', 22)
(4, 'Adam', 'Nguyen', 20)
(7, 'Julie', 'Roux', 22)
(8, 'Milan', 'Zoric', 19)
(11, 'Omar', 'Benali', 22)
(12, 'Chloe', 'Fabre', 21)
(13, 'Lea', 'Giraud', 20)
(16, 'Aline', 'Marchand', 22)
(20, 'Nina', 'Kovac', 22)
(22, 'Ibrahim', 'Sow', 21)
(24, 'Anna', 'Volkova', 20)
(26, 'Ava', 'Johnson', 22)
(28, 'Mason', 'Wilson', 21)
(29, 'Mia', 'Brown', 20)
(30, 'Evelyn', 'Jones', 19)
(34, 'Oliver', 'Clark', 22)
(36, 'Lucas', 'Walker', 20)
(37, 'Emily', 'Hall', 21)
(40, 'Grace', 'Wright', 22)


In [63]:
cursor.execute(""" UPDATE young_players 
                   SET age=31 
                   WHERE first_name = 'Luca'
               """)

for table in cursor.fetchall():
    print(table)

WithCheckOptionViolation: new row violates check option for view "young_players"
DETAIL:  Failing row contains (1, Luca, Moretti, luca.moretti@nba.com, M, 31, Guard, 14.20, Lyon, ARA, 1).


In [73]:
cursor.execute(""" UPDATE player 
                   SET age=31 
                   WHERE first_name = 'Luca'
               """)


In [74]:
cursor.execute("""SELECT * 
                  FROM young_players  
               """)

for table in cursor.fetchall():
    print(table)

(3, 'Ethan', 'Durand', 22)
(4, 'Adam', 'Nguyen', 20)
(7, 'Julie', 'Roux', 22)
(8, 'Milan', 'Zoric', 19)
(11, 'Omar', 'Benali', 22)
(12, 'Chloe', 'Fabre', 21)
(13, 'Lea', 'Giraud', 20)
(16, 'Aline', 'Marchand', 22)
(20, 'Nina', 'Kovac', 22)
(22, 'Ibrahim', 'Sow', 21)
(24, 'Anna', 'Volkova', 20)
(26, 'Ava', 'Johnson', 22)
(28, 'Mason', 'Wilson', 21)
(29, 'Mia', 'Brown', 20)
(30, 'Evelyn', 'Jones', 19)
(34, 'Oliver', 'Clark', 22)
(36, 'Lucas', 'Walker', 20)
(37, 'Emily', 'Hall', 21)
(40, 'Grace', 'Wright', 22)


### 5. Create a team_roster view containing the details of the players in the 'Lyon Eagles' team

In [77]:
cursor.execute("""CREATE OR REPLACE VIEW team_roaster AS
                  SELECT p.player_id, p.first_name, p.last_name, p.age, p.position, t.name as team_name
                  FROM player p JOIN team t ON p.team_id=t.team_id
                  WHERE t.name = 'Lyon Eagles'
               """)

In [78]:
cursor.execute("""SELECT * 
                  FROM team_roaster  
               """)

for table in cursor.fetchall():
    print(table)

(1, 'Luca', 'Moretti', 21, 'Guard', 'Lyon Eagles')
(2, 'Noah', 'Martin', 24, 'Forward', 'Lyon Eagles')
(3, 'Ethan', 'Durand', 22, 'Guard', 'Lyon Eagles')
(4, 'Adam', 'Nguyen', 20, 'Center', 'Lyon Eagles')
(5, 'Sofia', 'Bernard', 23, 'Forward', 'Lyon Eagles')
(6, 'Amine', 'Khelifi', 25, 'Guard', 'Lyon Eagles')
(7, 'Julie', 'Roux', 22, 'Coach Assistant', 'Lyon Eagles')
(8, 'Milan', 'Zoric', 19, 'Guard', 'Lyon Eagles')


In [79]:
# [information_schema.tables] keep listing of every table being managed by Postgres for a particular database.
# specifying the tabel_schema to 'public' to only list tables that you create.
cursor.execute("""SELECT table_name 
                  FROM information_schema.views 
                  WHERE table_schema = 'public'  
               """)

for table in cursor.fetchall():
    print(table)

('team_roaster',)


## Exercise 2

### What are the different player positions? The list must not contain duplicates.

In [80]:
cursor.execute("""SELECT DISTINCT position 
                  FROM player  
               """)

for table in cursor.fetchall():
    print(table)

('Forward',)
('Center',)
('Guard',)
('Coach Assistant',)


### 2. Which players belong to a team whose city name ends with 'ville'? Sort the result in descending order by age.

In [81]:
cursor.execute("""SELECT p.first_name, p.last_name, p.age, t.city
                  FROM player p JOIN team t ON p.team_id = t.team_id
                  WHERE t.city ILIKE '%ville'
                  ORDER BY p.age DESC
               """)

for table in cursor.fetchall():
    print(table)

('James', 'King', 27, 'Greenville')
('Daniel', 'Young', 26, 'Greenville')
('Henry', 'Taylor', 25, 'Shelbyville')
('Logan', 'Davis', 24, 'Shelbyville')
('Caleb', 'Harris', 24, 'Greenville')
('Jack', 'Miller', 23, 'Shelbyville')
('Zoe', 'Anderson', 23, 'Shelbyville')
('Isabella', 'Lewis', 23, 'Greenville')
('Ava', 'Johnson', 22, 'Shelbyville')
('Oliver', 'Clark', 22, 'Greenville')
('Grace', 'Wright', 22, 'Greenville')
('Emily', 'Hall', 21, 'Greenville')
('Mason', 'Wilson', 21, 'Shelbyville')
('Lucas', 'Walker', 20, 'Greenville')
('Mia', 'Brown', 20, 'Shelbyville')
('Evelyn', 'Jones', 19, 'Shelbyville')


### 3. Who are the players coached by 'Jean Dupont'? Sort the result by team name.

In [82]:
cursor.execute("""SELECT p.first_name, p.last_name, p.age, t.city
                  FROM player p 
                      JOIN team t ON p.team_id = t.team_id
                      JOIN coach c ON t.team_id = c.team_id
                  WHERE c.full_name = 'Jean Dupont'
                  ORDER BY t.name
               """)

for table in cursor.fetchall():
    print(table)

('Luca', 'Moretti', 21, 'Lyon')
('Noah', 'Martin', 24, 'Lyon')
('Ethan', 'Durand', 22, 'Lyon')
('Adam', 'Nguyen', 20, 'Lyon')
('Sofia', 'Bernard', 23, 'Lyon')
('Amine', 'Khelifi', 25, 'Lyon')
('Julie', 'Roux', 22, 'Lyon')
('Milan', 'Zoric', 19, 'Lyon')


### What is the minimum and maximum age of players?

In [83]:
cursor.execute("""SELECT MIN(age) as min_age, MAX(age) as max_age
                  FROM player  
               """)

for table in cursor.fetchall():
    print(table)

(19, 28)


### Which players are neither 'Forward', nor 'Center', nor 'Coach Assistant'? Hint: Write the query using subqueries.

In [93]:
# building the removal set P-Q. The set Q is
cursor.execute(""" SELECT * 
                   FROM player p  
                   WHERE p.position = 'Forward' OR p.position='Center' OR p.position = 'Coach Assistant'  
               """)

for table in cursor.fetchall():
    print(table)

(2, 'Noah', 'Martin', 'noah.martin@gmail.com', 'M', 24, 'Forward', Decimal('11.30'), 'Lyon', 'ARA', 1)
(4, 'Adam', 'Nguyen', 'adam.nguyen@outlook.com', 'M', 20, 'Center', Decimal('6.50'), 'Lyon', 'ARA', 1)
(5, 'Sofia', 'Bernard', 'sofia.bernard@gmail.com', 'F', 23, 'Forward', Decimal('12.90'), 'Lyon', 'ARA', 1)
(7, 'Julie', 'Roux', 'julie.roux@gmail.com', 'F', 22, 'Coach Assistant', Decimal('0.00'), 'Lyon', 'ARA', 1)
(9, 'Yanis', 'Haddad', 'yanis.haddad@nba.com', 'M', 26, 'Forward', Decimal('13.10'), 'Marseille', 'PACA', 2)
(11, 'Omar', 'Benali', 'omar.benali@yahoo.com', 'M', 22, 'Center', Decimal('7.90'), 'Aubagne', 'PACA', 2)
(12, 'Chloe', 'Fabre', 'chloe.fabre@gmail.com', 'F', 21, 'Forward', Decimal('9.50'), 'Marseille', 'PACA', 2)
(15, 'Boris', 'Ivanov', 'boris.ivanov@nba.com', 'M', 27, 'Center', Decimal('5.40'), 'Marseille', 'PACA', 2)
(16, 'Aline', 'Marchand', 'aline.marchand@gmail.com', 'F', 22, 'Coach Assistant', Decimal('0.00'), 'Marseille', 'PACA', 2)
(18, 'Anton', 'Smirnov',

In [94]:
# A better version is
cursor.execute(""" SELECT * 
                   FROM player p  
                   WHERE p.position IN ('Forward' , 'Center' , 'Coach Assistant')
               """)

for table in cursor.fetchall():
    print(table)

(2, 'Noah', 'Martin', 'noah.martin@gmail.com', 'M', 24, 'Forward', Decimal('11.30'), 'Lyon', 'ARA', 1)
(4, 'Adam', 'Nguyen', 'adam.nguyen@outlook.com', 'M', 20, 'Center', Decimal('6.50'), 'Lyon', 'ARA', 1)
(5, 'Sofia', 'Bernard', 'sofia.bernard@gmail.com', 'F', 23, 'Forward', Decimal('12.90'), 'Lyon', 'ARA', 1)
(7, 'Julie', 'Roux', 'julie.roux@gmail.com', 'F', 22, 'Coach Assistant', Decimal('0.00'), 'Lyon', 'ARA', 1)
(9, 'Yanis', 'Haddad', 'yanis.haddad@nba.com', 'M', 26, 'Forward', Decimal('13.10'), 'Marseille', 'PACA', 2)
(11, 'Omar', 'Benali', 'omar.benali@yahoo.com', 'M', 22, 'Center', Decimal('7.90'), 'Aubagne', 'PACA', 2)
(12, 'Chloe', 'Fabre', 'chloe.fabre@gmail.com', 'F', 21, 'Forward', Decimal('9.50'), 'Marseille', 'PACA', 2)
(15, 'Boris', 'Ivanov', 'boris.ivanov@nba.com', 'M', 27, 'Center', Decimal('5.40'), 'Marseille', 'PACA', 2)
(16, 'Aline', 'Marchand', 'aline.marchand@gmail.com', 'F', 22, 'Coach Assistant', Decimal('0.00'), 'Marseille', 'PACA', 2)
(18, 'Anton', 'Smirnov',

In [99]:
# P 
cursor.execute(""" SELECT * 
                   FROM player p2 --- SET P 
                   WHERE p2.position NOT IN (SELECT DISTINCT position 
                                             FROM player p  
                                             WHERE p.position IN ('Forward' , 'Center' , 'Coach Assistant'))
               """)

for table in cursor.fetchall():
    print(table)

(1, 'Luca', 'Moretti', 'luca.moretti@nba.com', 'M', 21, 'Guard', Decimal('14.20'), 'Lyon', 'ARA', 1)
(3, 'Ethan', 'Durand', 'ethan.durand@yahoo.com', 'M', 22, 'Guard', Decimal('9.70'), 'Villeurbanne', 'ARA', 1)
(6, 'Amine', 'Khelifi', 'amine.khelifi@nba.com', 'M', 25, 'Guard', Decimal('8.40'), 'Bron', 'ARA', 1)
(8, 'Milan', 'Zoric', 'milan.zoric@nba.com', 'M', 19, 'Guard', Decimal('7.20'), 'Lyon', 'ARA', 1)
(10, 'Mehdi', 'Talbi', 'mehdi.talbi@gmail.com', 'M', 24, 'Guard', Decimal('10.80'), 'Marseille', 'PACA', 2)
(13, 'Lea', 'Giraud', 'lea.giraud@nba.com', 'F', 20, 'Guard', Decimal('6.80'), 'Marseille', 'PACA', 2)
(14, 'Hugo', 'Perez', 'hugo.perez@hotmail.com', 'M', 23, 'Guard', Decimal('8.60'), 'Marseille', 'PACA', 2)
(17, 'Pierre', 'Leclerc', 'pierre.leclerc@nba.com', 'M', 28, 'Guard', Decimal('15.20'), 'Paris', 'IDF', 3)
(21, 'Elise', 'Laurent', 'elise.laurent@nba.com', 'F', 25, 'Guard', Decimal('7.70'), 'Paris', 'IDF', 3)
(22, 'Ibrahim', 'Sow', 'ibrahim.sow@hotmail.com', 'M', 21, '

### Who is the youngest player contained in the database?  Display their name, age, position, and average points.

In [111]:
cursor.execute(""" SELECT p.first_name, p.last_name, p.age, p.position, AVG(s.points) as avg_points
                   FROM player p JOIN match_player s ON p.player_id = s.player_id
                   GROUP BY p.first_name, p.last_name, p.age, p.position
                   ORDER BY age ASC
                   LIMIT 1
               """)

for table in cursor.fetchall():
    print(table)

('Evelyn', 'Jones', 19, 'Guard', Decimal('11.5000000000000000'))


In [112]:
cursor.execute(""" SELECT p.first_name, p.last_name, p.age, p.position, AVG(s.points) as avg_points
                   FROM player p JOIN match_player s ON p.player_id = s.player_id
                   WHERE p.age = (SELECT MIN(age) FROM player)
                   GROUP BY p.first_name, p.last_name, p.age, p.position
               """)

for table in cursor.fetchall():
    print(table)

('Evelyn', 'Jones', 19, 'Guard', Decimal('11.5000000000000000'))
('Milan', 'Zoric', 19, 'Guard', Decimal('13.6666666666666667'))


### 7. What is the average number of points scored by players who play as 'Forward'?

In [115]:
cursor.execute(""" SELECT p.first_name, p.last_name, p.age, p.position, AVG(s.points) as avg_points
                   FROM player p JOIN match_player s ON p.player_id = s.player_id
                   WHERE p.position = 'Forward'
                   GROUP BY p.first_name, p.last_name, p.age, p.position
               """)

for table in cursor.fetchall():
    print(table)

('Anton', 'Smirnov', 24, 'Forward', Decimal('14.5000000000000000'))
('Ava', 'Johnson', 22, 'Forward', Decimal('11.5000000000000000'))
('Chloe', 'Fabre', 21, 'Forward', Decimal('11.6666666666666667'))
('Emily', 'Hall', 21, 'Forward', Decimal('12.0000000000000000'))
('Mia', 'Brown', 20, 'Forward', Decimal('11.5000000000000000'))
('Nina', 'Kovac', 22, 'Forward', Decimal('14.5000000000000000'))
('Noah', 'Martin', 24, 'Forward', Decimal('13.6666666666666667'))
('Oliver', 'Clark', 22, 'Forward', Decimal('12.0000000000000000'))
('Sofia', 'Bernard', 23, 'Forward', Decimal('13.6666666666666667'))
('Yanis', 'Haddad', 26, 'Forward', Decimal('11.6666666666666667'))


### 8. Calculate the number of players in the league and the total average points per game for each position.

In [116]:
cursor.execute(""" SELECT p.position, COUNT(p.player_id), AVG(s.points) as avg_points
                   FROM player p JOIN match_player s ON p.player_id = s.player_id
                   GROUP BY p.position
               """)

for table in cursor.fetchall():
    print(table)

('Forward', 24, Decimal('12.6666666666666667'))
('Guard', 39, Decimal('12.7435897435897436'))


### Wait! No center Scored? Sounds weird!

In [120]:
update_script = """
--- there are no centers, use the script below to update the database with center scores ---

-- Centers score in existing matches; safe to re-run.
WITH center_points(match_date, team_name, first_name, last_name, minutes, points) AS (
  VALUES
    -- 2025-01-15 Lyon vs Paris
    ('2025-01-15','Lyon Eagles','Adam','Nguyen',18,10),
    ('2025-01-15','Paris Titans','Rayan','Diallo',20,8),

    -- 2025-01-22 Marseille vs Greenville
    ('2025-01-22','Marseille Sharks','Omar','Benali',18,9),
    ('2025-01-22','Greenville Wolves','James','King',16,7),

    -- 2025-02-03 Paris vs Shelbyville
    ('2025-02-03','Paris Titans','Paul','Garcia',19,12),
    ('2025-02-03','Shelbyville Bears','Logan','Davis',20,8),

    -- 2025-02-17 Shelbyville vs Lyon
    ('2025-02-17','Shelbyville Bears','Henry','Taylor',18,7),
    ('2025-02-17','Lyon Eagles','Adam','Nguyen',20,9),

    -- 2025-03-01 Greenville vs Marseille
    ('2025-03-01','Greenville Wolves','Isabella','Lewis',17,6),
    ('2025-03-01','Marseille Sharks','Boris','Ivanov',16,8),

    -- 2025-03-10 Lyon vs Marseille
    ('2025-03-10','Lyon Eagles','Adam','Nguyen',18,7),
    ('2025-03-10','Marseille Sharks','Omar','Benali',18,9)
)
INSERT INTO match_player (match_id, player_id, starter, minutes, points)
SELECT m.match_id, p.player_id, FALSE, cp.minutes, cp.points
FROM center_points cp
JOIN team t ON t.name = cp.team_name
JOIN player p
  ON p.team_id = t.team_id
 AND p.first_name = cp.first_name
 AND p.last_name  = cp.last_name
 AND p.position   = 'Center'
JOIN match m
  ON m.match_date = cp.match_date
 AND (m.home_team_id = t.team_id OR m.away_team_id = t.team_id)
ON CONFLICT DO NOTHING;

-- Recompute match totals from match_player so scores stay consistent (idempotent).

WITH sums AS (
  SELECT
    m.match_id,
    SUM(CASE WHEN pl.team_id = m.home_team_id THEN mp.points ELSE 0 END) AS home_pts,
    SUM(CASE WHEN pl.team_id = m.away_team_id THEN mp.points ELSE 0 END) AS away_pts
  FROM match m
  JOIN match_player mp ON mp.match_id = m.match_id
  JOIN player pl ON pl.player_id = mp.player_id
  GROUP BY m.match_id
)
UPDATE match m
SET home_score = s.home_pts,
    away_score = s.away_pts
FROM sums s
WHERE m.match_id = s.match_id;


"""

In [121]:
cursor.execute(update_script)

### Trying query 8 after the update

In [122]:
cursor.execute(""" SELECT p.position, COUNT(p.player_id), AVG(s.points) as avg_points
                   FROM player p JOIN match_player s ON p.player_id = s.player_id
                   GROUP BY p.position
               """)

for table in cursor.fetchall():
    print(table)

('Forward', 24, Decimal('12.6666666666666667'))
('Center', 12, Decimal('8.3333333333333333'))
('Guard', 39, Decimal('12.7435897435897436'))


### 9. Modify the previous query to get the result also by team in each position.

In [126]:
cursor.execute(""" SELECT t.name AS team, p.position, COUNT(p.player_id), AVG(s.points) as avg_points
                   FROM player p 
                         JOIN match_player s ON p.player_id = s.player_id
                         JOIN team t ON p.team_id = t.team_id
                   GROUP BY p.position, t.name
               """)

for table in cursor.fetchall():
    print(table)

('Greenville Wolves', 'Forward', 4, Decimal('12.0000000000000000'))
('Lyon Eagles', 'Guard', 12, Decimal('13.6666666666666667'))
('Marseille Sharks', 'Forward', 6, Decimal('11.6666666666666667'))
('Shelbyville Bears', 'Guard', 6, Decimal('11.5000000000000000'))
('Lyon Eagles', 'Forward', 6, Decimal('13.6666666666666667'))
('Marseille Sharks', 'Guard', 9, Decimal('11.6666666666666667'))
('Greenville Wolves', 'Guard', 6, Decimal('12.0000000000000000'))
('Paris Titans', 'Guard', 6, Decimal('14.5000000000000000'))
('Paris Titans', 'Forward', 4, Decimal('14.5000000000000000'))
('Shelbyville Bears', 'Forward', 4, Decimal('11.5000000000000000'))


### 10. Calculate the number of players per team. Display also the team name and coach name.

In [129]:
cursor.execute(""" SELECT t.name AS team, c.full_name AS coach, COUNT(p.player_id)
                   FROM  player p 
                         JOIN team t ON p.team_id = t.team_id
                         JOIN coach c ON c.team_id = t.team_id
                   GROUP BY c.full_name, t.name
               """)

for table in cursor.fetchall():
    print(table)

('Paris Titans', 'Thomas Leroy', 8)
('Marseille Sharks', 'Amina Bensaid', 8)
('Greenville Wolves', 'Marco Rossi', 8)
('Lyon Eagles', 'Jean Dupont', 8)
('Shelbyville Bears', 'Clara Moreau', 8)


## Exercise 3

### 1. Show the average number of points for each team with at least 8 players and order the results by average points in descending order.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 2. Find players whose average points are higher than the average points in their team.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 3. List players who play in 'Lyon' or 'Marseille'.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 4. Find players who are not assigned to any match.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 5. Find players whose average points are above the overall league average.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 6. Find players who have played in at least one match coached by 'Jean Dupont'

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

###  7. Show 5 player names in uppercase and how long their email address is

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

### 8. Rank players within each team based on their average points, with the highest scorer getting rank 1.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)

###  9. Find players whose email ends with '@nba.com'.

In [ ]:
cursor.execute(""" YOUR QUERY HERE!!! """)

for table in cursor.fetchall():
    print(table)